In [1]:
# 커널 확인
print("hello")

hello


# lenet 5 구현하기

In [2]:
# lenet5 구현하기
# lib import
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [3]:
data_transform = transforms.Compose([
    transforms.ToTensor(), # 이미지를 텐서로 변환
    transforms.Resize((32, 32)), # 이미지 크기를 32x32로 조정 (lenet5 입력 크기에 맞추기)
    transforms.Normalize((0.1307,), (0.3081,)) # MNIST 데이터셋의 평균과 표준편차로 정규화
])
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=data_transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=data_transform)
mnist_train



Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Resize(size=(32, 32), interpolation=bilinear, max_size=None, antialias=True)
               Normalize(mean=(0.1307,), std=(0.3081,))
           )

In [4]:
# 이미지 전처리, 셔플링, 배치 조정
from torch.utils.data import DataLoader
train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True) # 배치 크기 64로 설정, 데이터 셔플
test_loader = DataLoader(mnist_test, batch_size=64, shuffle=False) # 테스트 데이터는 섞지 않음


In [5]:
# 데이터 확인 (배치 크기, 채널 수, 가로, 세로)
next(iter(train_loader))[0].shape # train_loader에서 첫 번째 배치를 가져와서 확인

torch.Size([64, 1, 32, 32])

In [6]:
class LeNet5(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=0)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, kernel_size=2, stride=2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, kernel_size=2, stride=2)
        x = torch.flatten(x, 1)  # (N, 16*5*5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [7]:
# 모델 인스턴스 생성
model = LeNet5()

# Input type (torch.cuda.FloatTensor) and weight type (torch.FloatTensor) should be the same
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [8]:
# torchsummary 실행
from torchsummary import summary
summary(model, input_size=(1, 32, 32))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1            [-1, 6, 28, 28]             156
            Conv2d-2           [-1, 16, 10, 10]           2,416
            Linear-3                  [-1, 120]          48,120
            Linear-4                   [-1, 84]          10,164
            Linear-5                   [-1, 10]             850
Total params: 61,706
Trainable params: 61,706
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.05
Params size (MB): 0.24
Estimated Total Size (MB): 0.29
----------------------------------------------------------------


In [9]:
# optimizer와 loss function 정의
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
epochs = 10

In [10]:
# 모델 학습, tqdm으로 진행 상황 표시, tenseboard로 손실 기록
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter(log_dir='./logs/lenet5')
count = 0
from tqdm import tqdm
for epoch in range(epochs):
    model.train()
    total_loss = 0
    train_loader = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', unit='batch')
    for batch_idx, (data, target) in enumerate(tqdm(train_loader)):
        data, target = data.to(device), target.to(device) # 데이터를 GPU로 이동
        optimizer.zero_grad() # 기울기 초기화
        output = model(data) # 모델에 입력 데이터 전달하여 예측값 계산
        loss = criterion(output, target) # 예측값과 실제 레이블 간의 손실 계산
        loss.backward() # 역전파를 통해 기울기 계산
        optimizer.step() # 가중치 업데이트
        train_loader.set_description(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}') # tqdm에 현재 배치 손실 표시
        total_loss += loss.item() # 배치 손실을 총 손실에 누적
        writer.add_scalar('Loss/Batch', loss.item(), count) # TensorBoard에 배치 손실 기록
        count += 1
    avg_loss = total_loss / len(train_loader) # 평균 손실 계산
    print(f'Epoch: {epoch+1}, Loss: {avg_loss:.4f}')

100%|██████████| 938/938 [00:28<00:00, 32.71it/s]


Epoch: 1, Loss: 0.2059


100%|██████████| 938/938 [00:27<00:00, 33.58it/s]


Epoch: 2, Loss: 0.0613


100%|██████████| 938/938 [00:25<00:00, 36.13it/s]


...

Epoch: 8, Loss: 0.0189


100%|██████████| 938/938 [00:26<00:00, 34.94it/s]


Epoch: 9, Loss: 0.0154


100%|██████████| 938/938 [00:28<00:00, 33.10it/s]

Epoch: 10, Loss: 0.0145


In [11]:
# 모델 평가
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = torch.max(output.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()
print(f'Accuracy: {100 * correct / total:.2f}%')


Accuracy: 98.92%


In [15]:
# loss 시각화
# TensorBoard 로그 디렉토리에서 손실 데이터를 읽어와서 시각화
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
writer = SummaryWriter(log_dir='./logs/lenet5')
loss_data = writer._get_scalar_data('Loss/Batch')
# 손실 데이터를 시각화
steps = [entry.step for entry in loss_data]
values = [entry.value for entry in loss_data]
plt.plot(steps, values)
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.title('Training Loss over Batches')
plt.show()




AttributeError: 'SummaryWriter' object has no attribute '_get_scalar_data'

In [ ]:
# cannot import name 'SummaryReader' from 'torch.utils.tensorboard'
